# Imports

In [2]:
import numpy as np
import plotly.express as px
%pip install nbformat


[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: /opt/homebrew/opt/python@3.10/bin/python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Defining the Input Signal

In [84]:
L = 12 # Pick 10, 11 or 12
N = 2**L

# Frequencies:
def ang_vel(frequency):
    return (2*np.pi*frequency)/N

w_1=ang_vel(50)
w_2=ang_vel(199)
w_3=ang_vel(478)

n = np.arange(N) # create array for [1, N-1]

xreal=np.sin(w_1*n)+np.cos(w_2*n)+np.sin(w_3*n) # Signal defined

x= xreal.astype(complex) # Full signal

# Plotting the Signal

In [81]:
px.line(y=xreal)

#  New DFT Weights and DFT Functions

In [85]:
def weight (N, k, n):
    return np.exp(-1j*2*np.pi*k*n/N)

# DFT being x[k]=1/N*sum_{n=0}^{N-1} X[n] * -weight(N, k, n)

print(f"Input signal x[n]: {x}")

def DFT(x):
    N = len(x)
    X = np.zeros(N).astype(complex)
    for k in range(N):
        X[k] = np.sum(x * weight(N, k, n)) # Just keep adding to X[k]
    return X/N

d=DFT(x)
d[np.abs(d) < 1e-10]=0

print(f"DFT: {d}")

%timeit DFT(x)

Input signal x[n]: [ 1.        +0.j  1.69967464+0.j  1.96670928+0.j ... -0.42756125+0.j
 -0.32801424+0.j  0.20786174+0.j]
DFT: [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
286 ms ± 14.7 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


## Plot DFT

In [86]:
px.line(y=np.abs(d))

### Plot Angle

In [67]:

px.line(y=np.angle(d))

# Fast Fourier Transform

In [79]:
def FFT(x):
    N = len(x)
    if N <= 1: # End of recursion
        return x
    even=FFT(x[0::2]) # Even indices
    odd=FFT(x[1::2]) # Odd indices
    T = [weight(N,k,1)*odd[k] for k in range(N//2)] # Twiddle factor
    return np.concatenate([even + T, even - T]) # Combine

print(f"Input: {x}")
fft=FFT(x)
fft[np.abs(fft) < 1e-10]=0 # Cleaning Up
print(f"FFT: {fft}")

%timeit FFT(x)


Input: [ 1.        +0.j  1.69967464+0.j  1.96670928+0.j ... -0.42756125+0.j
 -0.32801424+0.j  0.20786174+0.j]
FFT: [0.+0.j 0.+0.j 0.+0.j ... 0.+0.j 0.+0.j 0.+0.j]
29.2 ms ± 849 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Plot FFT

In [76]:
px.line(y=np.abs(fft))